In [14]:
# %pip install scikit-learn

In [15]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

In [16]:
# endereço atual
base = r"C:\Users\vini7\Desktop\Arquivos Python\Estudo de Caso Lighthouse 2026\Estudo-de-Caso-Lighthouse-2026\data"

# Carrega cada csv
orders = pd.read_csv(base + r"\orders.csv")
order_items = pd.read_csv(base + r"\order_items.csv")
products = pd.read_csv(base + r"\products.csv")
product_variants = pd.read_csv(base + r"\product_variants.csv")

In [17]:
# Dataset unificado com todos os dados necessários
dataset = (
    order_items
    .merge(
        product_variants,
        left_on="product_variant_id",
        right_on="id",
        how="inner"
    )
    .merge(
        products,
        left_on="product_id",
        right_on="id",
        how="inner"
    )
    .drop(columns="id")
    .merge(
        orders,
        left_on="order_id",
        right_on="id",
        how="inner"
    )
    .drop(columns="id")
)


# # Selecionando somente vendas pagas e confirmadas
dataset = dataset [
    dataset["status"].isin(["paid", "confirmed"])
]

dataset

,id_x,order_id,product_variant_id,quantity,unit_price,icms_rate_x,ipi_rate_x,line_total,id_y,product_id,...,customer_id,salesperson_id,location_id,status,subtotal,discount_amount,total,placed_at,created_at,updated_at
0,1,1,113,6,53.89,12.0,0.0,323.34,113,59,...,1136,NaN,1,paid,323.34,35.57,287.77,2022-09-06 05:37:37,2022-09-06 05:37:37,2022-09-06 05:37:37
1,2,2,293,9,2398.41,17.0,10.0,21585.69,293,146,...,618,9.0,4,paid,53199.05,0.00,53199.05,2023-02-03 04:36:21,2023-02-03 04:36:21,2023-02-03 04:36:21
2,3,2,366,2,2697.71,17.0,10.0,5395.42,366,180,...,618,9.0,4,paid,53199.05,0.00,53199.05,2023-02-03 04:36:21,2023-02-03 04:36:21,2023-02-03 04:36:21
3,4,2,561,1,3757.73,18.0,0.0,3757.73,561,275,...,618,9.0,4,paid,53199.05,0.00,53199.05,2023-02-03 04:36:21,2023-02-03 04:36:21,2023-02-03 04:36:21
4,5,2,385,5,849.69,18.0,10.0,4248.45,385,190,...,618,9.0,4,paid,53199.05,0.00,53199.05,2023-02-03 04:36:21,2023-02-03 04:36:21,2023-02-03 04:36:21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
147315,150317,49999,199,5,1845.98,18.0,10.0,9229.90,199,97,...,1970,NaN,1,paid,27434.83,0.00,27434.83,2026-03-07 17:24:39,2026-03-07 17:24:39,2026-03-07 17:24:39
147316,150318,49999,619,9,988.70,12.0,10.0,8898.30,619,303,...,1970,NaN,1,paid,27434.83,0.00,27434.83,2026-03-07 17:24:39,2026-03-07 17:24:39,2026-03-07 17:24:39
147317,150319,50000,219,4,2189.02,12.0,5.0,8756.08,219,108,...,236,NaN,4,paid,51275.86,4614.83,46661.03,2020-04-13 17:35:27,2020-04-13 17:35:27,2020-04-13 17:35:27
147318,150320,50000,305,9,3102.51,12.0,5.0,27922.59,305,152,...,236,NaN,4,paid,51275.86,4614.83,46661.03,2020-04-13 17:35:27,2020-04-13 17:35:27,2020-04-13 17:35:27


In [19]:
# Mantém somente a coluna de cliente e produto e remove duplicadas
compras = dataset[
    ["customer_id", "product_id"]
].drop_duplicates()

# Matriz Usuário x Produto
matriz = pd.crosstab(
    compras["customer_id"],
    compras["product_id"]
)

# Faz a transposição da matriz
matriz_produtos = matriz.T

matriz_produtos

customer_id,1,2,3,4,5,6,7,8,9,10,...,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000
product_id,,,,,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,1,0,0,1,0,...,1,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,1,0,...,1,0,0,1,0,0,0,0,0,0
3,0,0,0,0,1,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,1,0,0,0,0,0,0,...,1,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
496,0,0,0,0,0,0,0,1,0,0,...,1,0,0,0,0,0,0,0,0,1
497,0,0,0,0,1,0,0,1,0,1,...,0,0,1,0,1,0,0,1,0,1
498,0,0,0,0,0,0,0,1,1,0,...,0,0,0,1,0,0,0,0,0,0


In [22]:
# Calcula a similaridade de cosseno
similarity = cosine_similarity(matriz_produtos)

# Criação de um dataframe com as similaridades
similarity_df = pd.DataFrame(
    similarity,
    index = matriz_produtos.index,
    columns= matriz_produtos.index
)

similarity_df

product_id,1,2,3,4,5,6,7,8,9,10,...,491,492,493,494,495,496,497,498,499,500
product_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.193265,0.146990,0.133071,0.200044,0.164224,0.139777,0.095806,0.150514,0.101290,...,0.146230,0.125136,0.183171,0.105112,0.088852,0.154516,0.146301,0.190438,0.078330,0.132362
2,0.193265,1.000000,0.143860,0.151775,0.206317,0.173275,0.143314,0.111096,0.176823,0.086222,...,0.100996,0.128800,0.195274,0.114419,0.079096,0.130671,0.173426,0.178608,0.112764,0.125579
3,0.146990,0.143860,1.000000,0.114027,0.146683,0.144072,0.144488,0.103699,0.168658,0.069435,...,0.097938,0.115170,0.178897,0.083542,0.101515,0.145153,0.184207,0.172681,0.083900,0.151225
4,0.133071,0.151775,0.114027,1.000000,0.166077,0.107511,0.126215,0.050996,0.117093,0.102438,...,0.089243,0.127433,0.141923,0.109770,0.068643,0.125882,0.143430,0.141390,0.080456,0.136504
5,0.200044,0.206317,0.146683,0.166077,1.000000,0.167692,0.125737,0.133871,0.189160,0.067229,...,0.087531,0.106482,0.171955,0.107331,0.095769,0.164792,0.152439,0.206806,0.119976,0.156497
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
496,0.154516,0.130671,0.145153,0.125882,0.164792,0.137751,0.117245,0.100664,0.105371,0.071368,...,0.082900,0.122458,0.173469,0.112701,0.115935,1.000000,0.133236,0.159739,0.126479,0.175887
497,0.146301,0.173426,0.184207,0.143430,0.152439,0.170686,0.146127,0.118425,0.159604,0.118943,...,0.108127,0.134877,0.187039,0.130664,0.085688,0.133236,1.000000,0.148159,0.094981,0.170724
498,0.190438,0.178608,0.172681,0.141390,0.206806,0.181902,0.147942,0.093831,0.179541,0.089006,...,0.109470,0.132959,0.155766,0.118114,0.086753,0.159739,0.148159,1.000000,0.070855,0.162043


In [ ]:
# Retorna somente o Motor de Popa 1949
motor_de_popa = products.loc[
    products["name"] == "Motor de Popa 1949",
    "id"
].iloc[0]

# Retorna os produtos similares em ordem de maior similaridade, excluindo o próprio motor_de_popa 
ranking = similarity_df[motor_de_popa].sort_values(ascending = False).drop(motor_de_popa)

ranking.head(5)

product_id
75     0.245200
295    0.229962
311    0.214818
308    0.212121
2      0.208824
Name: 180, dtype: float64

In [ ]:
# dataset com ID e nome dos produtos para cruzar com o dataset ranking
products_name = products[
    ["id", "name"]
].drop_duplicates().set_index("id")

# Retorna 
products_name_ranking = products_name.loc[ranking.index].copy()
products_name_ranking["similaridade"] = ranking.values

print(products_name_ranking)

                               name  similaridade
product_id                                       
75                 Vela Mestra 1913      0.245200
295               Cabo Náutico 2105      0.229962
311                GPS Plotter 2249      0.214818
308              Motor de Popa 1540      0.212121
2                  Vela Mestra 3870      0.208824
...                             ...           ...
105            Defensa Náutica 1200      0.065556
136         Colete Salva-Vidas 9822      0.064852
403                 Vela Mestra 807      0.061861
371              Motor de Popa 4088      0.059496
495           Sonar Transducer 4154      0.059496

[499 rows x 2 columns]
